In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parents[1]

DATA_FILE = PROJECT_ROOT / "data" / "processed" / "martin_selected_30_monthly_production_normalized.csv"
DCA_OUTPUT_DIR = PROJECT_ROOT / "reports" / "dca_outputs"

DATA_FILE.exists(), DCA_OUTPUT_DIR.exists()

In [ ]:
df = pd.read_csv(DATA_FILE)

df.head()

In [ ]:
id_columns = [
    "api8",
    "district",
    "lease_no",
    "well_no",
]

date_label_columns = [
    "first_prod_month",
    "cycle_year_month",
    "reported_first_month",
    "first_positive_prod_month",
]

numeric_columns = [
    "month_on_production",
    "oil_bbl",
    "casinghead_gas_mcf",
    "boe",
    "interval_length_proxy_ft",
    "reported_month_on_production",
]

for column in id_columns:
    df[column] = df[column].astype(str)

for column in date_label_columns:
    df[column] = df[column].astype(str)

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df.dtypes

In [ ]:
df = df.sort_values(["api8", "month_on_production"]).reset_index(drop=True)

df.head()

In [ ]:
actual_test_oil = (
    df[df["month_on_production"].between(25, 33)]
    .groupby(["api8", "lease_name", "well_no"], as_index=False)
    .agg(actual_test_oil_bbl=("oil_bbl", "sum"))
)

actual_test_oil.head()

In [ ]:
sorted(path.name for path in DCA_OUTPUT_DIR.glob("*.csv"))


In [ ]:
exponential_fit_summary = pd.read_csv(
    DCA_OUTPUT_DIR / "exponential_decline_fit_summary.csv",
    dtype={"api8": str},
)

hyperbolic_fit_summary = pd.read_csv(
    DCA_OUTPUT_DIR / "hyperbolic_decline_fit_summary.csv",
    dtype={"api8": str},
)

exponential_fit_summary.columns, hyperbolic_fit_summary.columns

In [ ]:
exponential_fit_summary.head()

In [ ]:
list(exponential_fit_summary.columns)

In [ ]:
list(hyperbolic_fit_summary.columns)

In [ ]:
exponential_fit_summary.shape, hyperbolic_fit_summary.shape

## Per-Well DCA Source

The per-well DCA results were created in the exponential DCA notebook.

This notebook will inspect that workflow and either:

- load an existing per-well output if it was saved somewhere else
- or recreate/export the per-well DCA comparison file from the same logic

In [ ]:
DCA_NOTEBOOK_DIR = PROJECT_ROOT / "notebooks" / "dca_workflow"

sorted(path.name for path in DCA_NOTEBOOK_DIR.glob("*exponential*"))

In [ ]:
exponential_script_path = DCA_NOTEBOOK_DIR / "02_exponential_decline_dca.py"

exponential_script_text = exponential_script_path.read_text(encoding="utf-8")

print(exponential_script_text[:3000])

In [ ]:
import json

exponential_notebook_path = DCA_NOTEBOOK_DIR / "02_exponential_decline_dca.ipynb"

exponential_notebook = json.loads(
    exponential_notebook_path.read_text(encoding="utf-8")
)

len(exponential_notebook["cells"])

In [ ]:
for cell_number, cell in enumerate(exponential_notebook["cells"]):
    if cell["cell_type"] == "code":
        source = "".join(cell["source"])
        first_line = source.strip().splitlines()[0] if source.strip() else ""
        print(cell_number, first_line)

In [ ]:
for cell_number in range(26, 34):
    cell = exponential_notebook["cells"][cell_number]
    print(f"\n--- Cell {cell_number} ({cell['cell_type']}) ---")
    print("".join(cell["source"]))

In [ ]:
for cell_number in range(1, 4):
    cell = exponential_notebook["cells"][cell_number]
    print(f"\n--- Cell {cell_number} ({cell['cell_type']}) ---")
    print("".join(cell["source"]))

In [ ]:
LEGACY_DCA_OUTPUT_DIR = Path(r"C:\Users\tawac\outputs")

LEGACY_DCA_OUTPUT_DIR.exists(), sorted(path.name for path in LEGACY_DCA_OUTPUT_DIR.glob("*.csv"))

In [ ]:
comparison_ready_files = [
    "per_well_exponential_dca_results.csv",
    "per_well_hyperbolic_comparison_ready.csv",
    "per_well_harmonic_comparison_ready.csv",
    "dca_all_models_per_well_comparison.csv",
    "dca_all_models_summary.csv",
    "dca_all_models_win_counts.csv",
]

for file_name in comparison_ready_files:
    source_path = LEGACY_DCA_OUTPUT_DIR / file_name
    destination_path = DCA_OUTPUT_DIR / file_name
    
    pd.read_csv(source_path, dtype={"api8": str}).to_csv(destination_path, index=False)

sorted(path.name for path in DCA_OUTPUT_DIR.glob("*.csv"))

In [ ]:
dca_all_models = pd.read_csv(
    DCA_OUTPUT_DIR / "dca_all_models_per_well_comparison.csv",
    dtype={"api8": str},
)

dca_all_models.head()

In [ ]:
list(dca_all_models.columns)


In [ ]:
dca_comparison_ready = dca_all_models[
    [
        "api8",
        "lease_name",
        "well_no",
        "actual_test_oil_bbl",
        "exponential_forecast_test_oil_bbl",
        "hyperbolic_forecast_test_oil_bbl",
        "harmonic_forecast_test_oil_bbl",
    ]
].copy()

dca_comparison_ready.head()

In [ ]:
dca_comparison_ready.to_csv(
    DCA_OUTPUT_DIR / "dca_comparison_ready_forecast_totals.csv",
    index=False,
)

In [ ]:
(DCA_OUTPUT_DIR / "dca_comparison_ready_forecast_totals.csv").exists()

## Export Result

This notebook located the legacy DCA per-well outputs that had been saved outside the project repo.

The key comparison files were copied into `reports/dca_outputs`.

A clean DCA comparison-ready forecast totals file was created:

- `dca_comparison_ready_forecast_totals.csv`

This file contains one row per well and includes:

- actual oil over months 25-33
- exponential DCA forecast oil over months 25-33
- hyperbolic DCA forecast oil over months 25-33
- harmonic DCA forecast oil over months 25-33

This output can now be merged with the fixed-origin ML results from notebook 07.

In [ ]:
DCA_COMPARISON_FILE = DCA_OUTPUT_DIR / "dca_comparison_ready_forecast_totals.csv"

dca_comparison_ready = pd.read_csv(
    DCA_COMPARISON_FILE,
    dtype={"api8": str},
)

dca_comparison_ready.head()

In [ ]:
list(dca_comparison_ready.columns)

In [ ]:
DCA_COMPARISON_FILE = DCA_OUTPUT_DIR / "dca_comparison_ready_forecast_totals.csv"

dca_comparison_ready = pd.read_csv(
    DCA_COMPARISON_FILE,
    dtype={"api8": str},
)

dca_comparison_ready.head()

In [ ]:
# Rebuild fixed-origin ML forecast rows if they are not already in memory.

modeling_df = df[df["month_on_production"].between(1, 24)].copy()

well_groups = modeling_df.groupby("api8", group_keys=False)

modeling_df["target_month_on_production"] = well_groups["month_on_production"].shift(-1)
modeling_df["target_next_oil_bbl"] = well_groups["oil_bbl"].shift(-1)
modeling_df["last_observed_oil_bbl"] = modeling_df["oil_bbl"]

modeling_df["trailing_3mo_avg_oil_bbl"] = well_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=3, min_periods=1).mean()
)

modeling_df["trailing_6mo_avg_oil_bbl"] = well_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=6, min_periods=1).mean()
)

train_rows = modeling_df.dropna(
    subset=[
        "target_month_on_production",
        "target_next_oil_bbl",
        "last_observed_oil_bbl",
    ]
).copy()

train_rows = train_rows[train_rows["target_month_on_production"].between(13, 24)].copy()

feature_columns = [
    "month_on_production",
    "last_observed_oil_bbl",
    "trailing_3mo_avg_oil_bbl",
    "trailing_6mo_avg_oil_bbl",
    "interval_length_proxy_ft",
]

target_column = "target_next_oil_bbl"

X_train = train_rows[feature_columns]
y_train = train_rows[target_column]

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

random_forest_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=4,
    random_state=42,
)
random_forest_model.fit(X_train, y_train)

origin_rows = df[df["month_on_production"].between(1, 24)].copy()

origin_groups = origin_rows.groupby("api8", group_keys=False)

origin_rows["trailing_3mo_avg_oil_bbl"] = origin_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=3, min_periods=1).mean()
)

origin_rows["trailing_6mo_avg_oil_bbl"] = origin_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=6, min_periods=1).mean()
)

month_24_rows = origin_rows[origin_rows["month_on_production"] == 24].copy()
month_24_rows["last_observed_oil_bbl"] = month_24_rows["oil_bbl"]

forecast_months = pd.DataFrame(
    {
        "target_month_on_production": list(range(25, 34))
    }
)

fixed_origin_rows = month_24_rows.merge(
    forecast_months,
    how="cross",
)

fixed_origin_rows["forecast_month_on_production"] = fixed_origin_rows[
    "target_month_on_production"
]

fixed_origin_X = fixed_origin_rows[
    [
        "forecast_month_on_production",
        "last_observed_oil_bbl",
        "trailing_3mo_avg_oil_bbl",
        "trailing_6mo_avg_oil_bbl",
        "interval_length_proxy_ft",
    ]
].copy()

fixed_origin_X = fixed_origin_X.rename(
    columns={
        "forecast_month_on_production": "month_on_production"
    }
)

fixed_origin_rows["linear_regression_forecast_oil_bbl"] = linear_model.predict(fixed_origin_X)
fixed_origin_rows["random_forest_forecast_oil_bbl"] = random_forest_model.predict(fixed_origin_X)

actual_rows = df[df["month_on_production"].between(25, 33)][
    [
        "api8",
        "month_on_production",
        "oil_bbl",
    ]
].copy()

actual_rows = actual_rows.rename(
    columns={
        "month_on_production": "target_month_on_production",
        "oil_bbl": "actual_oil_bbl",
    }
)

fixed_origin_rows = fixed_origin_rows.merge(
    actual_rows,
    on=["api8", "target_month_on_production"],
    how="left",
)

fixed_origin_rows["naive_fixed_origin_forecast_oil_bbl"] = fixed_origin_rows[
    "last_observed_oil_bbl"
]

fixed_origin_rows["trailing_3mo_fixed_origin_forecast_oil_bbl"] = fixed_origin_rows[
    "trailing_3mo_avg_oil_bbl"
]

fixed_origin_rows["trailing_6mo_fixed_origin_forecast_oil_bbl"] = fixed_origin_rows[
    "trailing_6mo_avg_oil_bbl"
]

ml_fixed_origin_totals = (
    fixed_origin_rows
    .groupby(["api8", "lease_name", "well_no"], as_index=False)
    .agg(
        actual_ml_test_oil_bbl=("actual_oil_bbl", "sum"),
        naive_fixed_origin_forecast_test_oil_bbl=("naive_fixed_origin_forecast_oil_bbl", "sum"),
        trailing_3mo_fixed_origin_forecast_test_oil_bbl=("trailing_3mo_fixed_origin_forecast_oil_bbl", "sum"),
        trailing_6mo_fixed_origin_forecast_test_oil_bbl=("trailing_6mo_fixed_origin_forecast_oil_bbl", "sum"),
        linear_regression_forecast_test_oil_bbl=("linear_regression_forecast_oil_bbl", "sum"),
        random_forest_forecast_test_oil_bbl=("random_forest_forecast_oil_bbl", "sum"),
    )
)

fixed_origin_rows.shape, ml_fixed_origin_totals.head()

In [ ]:
# Rebuild fixed-origin ML forecast rows if they are not already in memory.

modeling_df = df[df["month_on_production"].between(1, 24)].copy()

well_groups = modeling_df.groupby("api8", group_keys=False)

modeling_df["target_month_on_production"] = well_groups["month_on_production"].shift(-1)
modeling_df["target_next_oil_bbl"] = well_groups["oil_bbl"].shift(-1)
modeling_df["last_observed_oil_bbl"] = modeling_df["oil_bbl"]

modeling_df["trailing_3mo_avg_oil_bbl"] = well_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=3, min_periods=1).mean()
)

modeling_df["trailing_6mo_avg_oil_bbl"] = well_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=6, min_periods=1).mean()
)

train_rows = modeling_df.dropna(
    subset=[
        "target_month_on_production",
        "target_next_oil_bbl",
        "last_observed_oil_bbl",
    ]
).copy()

train_rows = train_rows[train_rows["target_month_on_production"].between(13, 24)].copy()

feature_columns = [
    "month_on_production",
    "last_observed_oil_bbl",
    "trailing_3mo_avg_oil_bbl",
    "trailing_6mo_avg_oil_bbl",
    "interval_length_proxy_ft",
]

target_column = "target_next_oil_bbl"

X_train = train_rows[feature_columns]
y_train = train_rows[target_column]

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

random_forest_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=4,
    random_state=42,
)
random_forest_model.fit(X_train, y_train)

origin_rows = df[df["month_on_production"].between(1, 24)].copy()

origin_groups = origin_rows.groupby("api8", group_keys=False)

origin_rows["trailing_3mo_avg_oil_bbl"] = origin_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=3, min_periods=1).mean()
)

origin_rows["trailing_6mo_avg_oil_bbl"] = origin_groups["oil_bbl"].transform(
    lambda values: values.rolling(window=6, min_periods=1).mean()
)

month_24_rows = origin_rows[origin_rows["month_on_production"] == 24].copy()
month_24_rows["last_observed_oil_bbl"] = month_24_rows["oil_bbl"]

forecast_months = pd.DataFrame(
    {
        "target_month_on_production": list(range(25, 34))
    }
)

fixed_origin_rows = month_24_rows.merge(
    forecast_months,
    how="cross",
)

fixed_origin_rows["forecast_month_on_production"] = fixed_origin_rows[
    "target_month_on_production"
]

fixed_origin_X = fixed_origin_rows[
    [
        "forecast_month_on_production",
        "last_observed_oil_bbl",
        "trailing_3mo_avg_oil_bbl",
        "trailing_6mo_avg_oil_bbl",
        "interval_length_proxy_ft",
    ]
].copy()

fixed_origin_X = fixed_origin_X.rename(
    columns={
        "forecast_month_on_production": "month_on_production"
    }
)

fixed_origin_rows["linear_regression_forecast_oil_bbl"] = linear_model.predict(fixed_origin_X)
fixed_origin_rows["random_forest_forecast_oil_bbl"] = random_forest_model.predict(fixed_origin_X)

actual_rows = df[df["month_on_production"].between(25, 33)][
    [
        "api8",
        "month_on_production",
        "oil_bbl",
    ]
].copy()

actual_rows = actual_rows.rename(
    columns={
        "month_on_production": "target_month_on_production",
        "oil_bbl": "actual_oil_bbl",
    }
)

fixed_origin_rows = fixed_origin_rows.merge(
    actual_rows,
    on=["api8", "target_month_on_production"],
    how="left",
)

fixed_origin_rows["naive_fixed_origin_forecast_oil_bbl"] = fixed_origin_rows[
    "last_observed_oil_bbl"
]

fixed_origin_rows["trailing_3mo_fixed_origin_forecast_oil_bbl"] = fixed_origin_rows[
    "trailing_3mo_avg_oil_bbl"
]

fixed_origin_rows["trailing_6mo_fixed_origin_forecast_oil_bbl"] = fixed_origin_rows[
    "trailing_6mo_avg_oil_bbl"
]

ml_fixed_origin_totals = (
    fixed_origin_rows
    .groupby(["api8", "lease_name", "well_no"], as_index=False)
    .agg(
        actual_ml_test_oil_bbl=("actual_oil_bbl", "sum"),
        naive_fixed_origin_forecast_test_oil_bbl=("naive_fixed_origin_forecast_oil_bbl", "sum"),
        trailing_3mo_fixed_origin_forecast_test_oil_bbl=("trailing_3mo_fixed_origin_forecast_oil_bbl", "sum"),
        trailing_6mo_fixed_origin_forecast_test_oil_bbl=("trailing_6mo_fixed_origin_forecast_oil_bbl", "sum"),
        linear_regression_forecast_test_oil_bbl=("linear_regression_forecast_oil_bbl", "sum"),
        random_forest_forecast_test_oil_bbl=("random_forest_forecast_oil_bbl", "sum"),
    )
)

fixed_origin_rows.shape, ml_fixed_origin_totals.head()

In [ ]:
print("ML columns:")
print(list(ml_fixed_origin_totals.columns))

print("\nDCA columns:")
print(list(dca_comparison_ready.columns))

print("\nML api8 sample:")
print(ml_fixed_origin_totals["api8"].head())

print("\nDCA api8 sample:")
print(dca_comparison_ready["api8"].head())

print("\nML rows:", ml_fixed_origin_totals.shape)
print("DCA rows:", dca_comparison_ready.shape)

In [ ]:
ml_vs_dca_totals = ml_fixed_origin_totals.merge(
    dca_comparison_ready.drop(columns=["lease_name", "well_no"]),
    on="api8",
    how="inner",
)

print("ML wells:", ml_fixed_origin_totals["api8"].nunique())
print("DCA wells:", dca_comparison_ready["api8"].nunique())
print("Merged wells:", ml_vs_dca_totals["api8"].nunique())
print("Merged shape:", ml_vs_dca_totals.shape)

ml_vs_dca_totals[
    [
        "api8",
        "lease_name",
        "well_no",
        "actual_test_oil_bbl",
        "actual_ml_test_oil_bbl",
        "naive_fixed_origin_forecast_test_oil_bbl",
        "linear_regression_forecast_test_oil_bbl",
        "random_forest_forecast_test_oil_bbl",
        "exponential_forecast_test_oil_bbl",
        "hyperbolic_forecast_test_oil_bbl",
        "harmonic_forecast_test_oil_bbl",
    ]
].head()

In [ ]:
def summarize_total_forecast(results_df, forecast_column, model_name):
    error = results_df[forecast_column] - results_df["actual_test_oil_bbl"]
    absolute_error = error.abs()
    
    return {
        "model": model_name,
        "mae_bbl": absolute_error.mean(),
        "bias_bbl": error.mean(),
        "wape": absolute_error.sum() / results_df["actual_test_oil_bbl"].sum(),
    }


ml_vs_dca_metrics = pd.DataFrame(
    [
        summarize_total_forecast(
            ml_vs_dca_totals,
            "naive_fixed_origin_forecast_test_oil_bbl",
            "naive_fixed_origin",
        ),
        summarize_total_forecast(
            ml_vs_dca_totals,
            "trailing_3mo_fixed_origin_forecast_test_oil_bbl",
            "trailing_3mo_fixed_origin",
        ),
        summarize_total_forecast(
            ml_vs_dca_totals,
            "trailing_6mo_fixed_origin_forecast_test_oil_bbl",
            "trailing_6mo_fixed_origin",
        ),
        summarize_total_forecast(
            ml_vs_dca_totals,
            "linear_regression_forecast_test_oil_bbl",
            "linear_regression_fixed_origin",
        ),
        summarize_total_forecast(
            ml_vs_dca_totals,
            "random_forest_forecast_test_oil_bbl",
            "random_forest_fixed_origin",
        ),
        summarize_total_forecast(
            ml_vs_dca_totals,
            "exponential_forecast_test_oil_bbl",
            "exponential_dca",
        ),
        summarize_total_forecast(
            ml_vs_dca_totals,
            "hyperbolic_forecast_test_oil_bbl",
            "hyperbolic_dca",
        ),
        summarize_total_forecast(
            ml_vs_dca_totals,
            "harmonic_forecast_test_oil_bbl",
            "harmonic_dca",
        ),
    ]
)

ml_vs_dca_metrics["wape_percent"] = ml_vs_dca_metrics["wape"] * 100

ml_vs_dca_metrics_display = (
    ml_vs_dca_metrics
    .sort_values("mae_bbl")
    .reset_index(drop=True)
)

ml_vs_dca_metrics_display.round(2)

In [ ]:
ML_OUTPUT_DIR = PROJECT_ROOT / "reports" / "ml_outputs"

ML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ml_vs_dca_totals.to_csv(
    ML_OUTPUT_DIR / "fixed_origin_ml_vs_dca_per_well_totals.csv",
    index=False,
)

ml_vs_dca_metrics_display.to_csv(
    ML_OUTPUT_DIR / "fixed_origin_ml_vs_dca_metrics.csv",
    index=False,
)

sorted(path.name for path in ML_OUTPUT_DIR.glob("*ml_vs_dca*.csv"))

## Final ML vs DCA Result

This notebook completed the fixed-origin comparison between simple ML methods and DCA methods.

All methods were compared on the same basis:

- forecast origin: month 24
- forecast window: months 25-33
- target: total oil production over months 25-33
- comparison level: one row per well

The best-performing method by MAE was linear regression.

Its MAE was approximately 6233 barrels over the full test window per well.

Its WAPE was approximately 19%.

This comparison is stricter than the rolling one-step-ahead notebook because production observations from months 25-33 were not used as ML input features.

The final comparison outputs were saved under `reports/ml_outputs`:

- `fixed_origin_ml_vs_dca_metrics.csv`
- `fixed_origin_ml_vs_dca_per_well_totals.csv`